## Project 3

## Relationship between broadband usage and Education

With the internet becoming so prominent in this century, I thought it would be nice to analyze the trend between education and internet usage. To do this, I looked at the education level of people in each state, and compared it to the bandwith usage of those states. My Hypothesis for this project is: States with lower broadband usage will have lower levels of education compared to states with higher broadband usage.

**Dataset(s) to be used:** 

[Dataset1]https://www.kaggle.com/datasets/tjkyner/bachelor-degree-majors-by-age-sex-and-state/data

[Dataset2]https://www.kaggle.com/datasets/thedevastator/us-broadband-usage-across-counties-and-zip-codes

**Analysis question:** 

How important is the access of internet with association to education?

**Columns that will (likely) be used:**

  [Dataset 1][Columns] Education Level, CD

  [Dataset 2][Columns] BroadBand Usage, ST

 **Columns to be used to merge/join them:**

  [Dataset 1] [column] ST

  [Dataset 2] [column] CD

**Hypothesis**: States with lower broadband usage will have lower levels of education compared to states with higher broadband usage.

## Library Imports and Loading in the DataFrames

In [2]:
# Some Library Imports
import pandas as pd
import plotly.express as px
from IPython.display import HTML

In [3]:
df_edu = pd.read_csv("education.csv")
df_broadband = pd.read_csv("broadband.csv")

## Cleaning Dataset 1: Education

#### What the DataFrame looks like with zero cleaning

In [4]:
df_edu

,Year,cd,Bachelors_degree_or_higher,high_school_or_some_degree,Less_than_high_school_graduate
0,2020,0_AK,121098,309698,33572
1,2020,0_DC,277816,177505,34652
2,2020,0_DE,175338,351177,57053
3,2020,0_ND,137958,303148,26631
4,2020,0_PR,121098,309698,33572
...,...,...,...,...,...
869,2021,9_PA,128094,391551,57616
870,2021,9_TN,184482,361428,63368
871,2021,9_TX,262849,478680,180807
872,2021,9_VA,113316,339930,48042


We are first cleaning the CD column as it is the main column we are working with later. My first step was to seperate the district code with the state as the original format had it formated as "District Code"_"State". Because all the rows followed this format, it made it a lot easier to work with where I could just split the column into two and use the underscore as a parameter.

In [5]:
# Cleaning up df_edu
df_edu["District Code"] = df_edu["cd"].str.split("_").str[0]
df_edu["State"] = df_edu["cd"].str.split("_").str[1]
df_edu = df_edu.drop(columns = "cd")

df_edu

,Year,Bachelors_degree_or_higher,high_school_or_some_degree,Less_than_high_school_graduate,District Code,State
0,2020,121098,309698,33572,0,AK
1,2020,277816,177505,34652,0,DC
2,2020,175338,351177,57053,0,DE
3,2020,137958,303148,26631,0,ND
4,2020,121098,309698,33572,0,PR
...,...,...,...,...,...,...
869,2021,128094,391551,57616,9,PA
870,2021,184482,361428,63368,9,TN
871,2021,262849,478680,180807,9,TX
872,2021,113316,339930,48042,9,VA


Now that we have a "district code" and "state" column, I began to work on cleaning it for visibility and to make my life easier when working with the data. I first moved the two newly created columns back into the front and renamed the other columns into something shorter. Doing this allows me to type less :D. 

In [6]:
# Moving District Code and State for better visibility
district_col = df_edu.pop("District Code")
state_col = df_edu.pop("State")

df_edu.insert(1,"District",district_col)
df_edu.insert(2,"State",state_col)

# Renaming columns to make it easier to work with in the future

df_edu = df_edu.rename(columns = {"Bachelors_degree_or_higher": ">= Bachelors",
                         "high_school_or_some_degree":"High School",
                         "Less_than_high_school_graduate":"< High School"})

df_edu

,Year,District,State,>= Bachelors,High School,< High School
0,2020,0,AK,121098,309698,33572
1,2020,0,DC,277816,177505,34652
2,2020,0,DE,175338,351177,57053
3,2020,0,ND,137958,303148,26631
4,2020,0,PR,121098,309698,33572
...,...,...,...,...,...,...
869,2021,9,PA,128094,391551,57616
870,2021,9,TN,184482,361428,63368
871,2021,9,TX,262849,478680,180807
872,2021,9,VA,113316,339930,48042


### Weighted Average

At this point in the project I ran into a bit of an issue. Because there was no formal "measure of education" for each state, I didn't really have a numerical value that could compare how much a state was compared to its counterparts. After a bit of thinking I decided to give different weights to the education columns. However, this is where the first assumption comes in.

**Assumption 1: Not all levels are created equal. For the sake of analysis each section will have a weight:**

Bachelors = 2

High school = 1

Less than high school = 0


Having a bachelors does not neccessarily make you "smarter" than someone who has less than a high school education. For the sake of my project, these numbers will be the predictors of high vs. low education.

After going through such assumptions, I worked on calculating the total and weighted average. The code below shows the additional two columns

In [7]:
# Weighted average calculation
df_edu["Weighted Average Sum"] = ((df_edu[">= Bachelors"] * 2 + df_edu["High School"] * 1 + df_edu["< High School"] * 0))
df_edu["Total"] = (df_edu[">= Bachelors"] + df_edu["High School"] + df_edu["< High School"])
df_edu["Weighted Average"] = (df_edu["Weighted Average Sum"] / df_edu["Total"]).round(3)

df_edu

,Year,District,State,>= Bachelors,High School,< High School,Weighted Average Sum,Total,Weighted Average
0,2020,0,AK,121098,309698,33572,551894,464368,1.188
1,2020,0,DC,277816,177505,34652,733137,489973,1.496
2,2020,0,DE,175338,351177,57053,701853,583568,1.203
3,2020,0,ND,137958,303148,26631,579064,467737,1.238
4,2020,0,PR,121098,309698,33572,551894,464368,1.188
...,...,...,...,...,...,...,...,...,...
869,2021,9,PA,128094,391551,57616,647739,577261,1.122
870,2021,9,TN,184482,361428,63368,730392,609278,1.199
871,2021,9,TX,262849,478680,180807,1004378,922336,1.089
872,2021,9,VA,113316,339930,48042,566562,501288,1.130


Now that the cleaning for the entire dataset has been completed, I created a quick sanity check to see that everything worked. The code below checks for the highest and lowest states. In this case, the variable names are a bit misleading as it is not necessarilly the states, but the districts inside the states.

In [8]:
# Check lowest weighted average
low_index = df_edu["Weighted Average"].idxmin()
low_state = df_edu["State"].iloc[low_index]
low_edu = df_edu["Weighted Average"].iloc[low_index]
# Check highest weighted average
high_index = df_edu["Weighted Average"].idxmax()
high_state = df_edu["State"].iloc[high_index]
high_edu = df_edu["Weighted Average"].iloc[high_index]

low_state,high_state
#Suprising, but we need to remember that these are not the states themselves, but districts in the different states.

('TX', 'IL')

### Analyzing a Specific State

From the mislabeling of the prior code, I realized I could take a look at education from a smaller scale. I decided I wanted to take a look at New York in the year 2021. Below is the code I used to filter it, but if anyone wants to see the education level for the other states, they can just change the State! :D

In [9]:
# Taking a look at New York for year 2021 Specifically
ny_df = df_edu[(df_edu["State"] == "NY") & (df_edu["Year"] == 2021)]
ny_df

,Year,District,State,>= Bachelors,High School,< High School,Weighted Average Sum,Total,Weighted Average
475,2021,1,NY,261740,332150,54326,855630,648216,1.320
496,2021,10,NY,391804,225037,87398,1008645,704239,1.432
509,2021,11,NY,121132,177081,34047,419345,332260,1.262
521,2021,12,NY,373927,121828,25878,869682,521633,1.667
531,2021,13,NY,114096,128938,49401,357130,292435,1.221
540,2021,14,NY,53977,93183,28705,201137,175865,1.144
547,2021,15,NY,160161,459012,177843,779334,797016,0.978
554,2021,16,NY,209806,191433,47627,611045,448866,1.361
560,2021,17,NY,185745,185739,38558,557229,410042,1.359
565,2021,18,NY,141865,254800,36374,538530,433039,1.244


below is just the code to run the choropleth. I used ChatGPT to assist in finding a working table to use.

In [10]:
# Creating a Choropleth Map for New York Education Distribution
import requests

district_url = "https://gisservices-np.its.ny.gov/arcgis/rest/services/NYS_Congressional_Districts/FeatureServer/0/query?where=1=1&outFields=*&f=geojson"
district_response = requests.get(district_url)
district_shapes = district_response.json()

In [29]:
# Plot
district_fig = px.choropleth_map(ny_df,
                           geojson = district_shapes,
                           locations = "District",
                           featureidkey = "properties.DISTRICT",
                           color = "Weighted Average",
                           center = {"lat": 42.9, "lon": -75.5},
                           zoom = 6,
                           height = 600,
                           title = "Weighted Average across New York Districts"
                           )
# district_fig.show()
HTML(district_fig.to_html(include_plotlyjs="cdn", full_html=False))

### Analyzing the Entire United States

Moving on to the main part, I began to work on the choropleth for the entire United States. I first needed to convert the state abbreviations into their full names. To do so, I took a dictionary of abbreviations and names from online and used it to update my state column :^).

In [12]:
abbr_to_name = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island",
    "SC": "South Carolina", "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas",
    "UT": "Utah", "VT": "Vermont", "VA": "Virginia", "WA": "Washington",
    "WV": "West Virginia", "WI": "Wisconsin", "WY": "Wyoming",
    "DC": "District of Columbia"
}


Looking at states specifically also meant that I needed to combine the districts. I grouped the counties together and took the mean of those numbers. I also filtered to only include 2020 as my other dataset was of 2020 information as well. And finally I dropped the District of Columbia as it is not really a state, and skewed my choropleth upwards.

In [13]:
# Looking at it from a bigger scale

states_df = df_edu[df_edu["Year"] == 2020]
states_df = pd.DataFrame(states_df.groupby("State")["Weighted Average"].mean()).reset_index()
states_df["State"] = states_df["State"].map(abbr_to_name)

states_df = states_df.dropna()
states_df = states_df.drop(index = 7).reset_index(drop=True)
states_df

,State,Weighted Average
0,Alaska,1.188000
1,Alabama,1.117571
2,Arkansas,1.124000
3,Arizona,1.110889
4,California,1.176038
5,Colorado,1.292250
6,Connecticut,1.298600
7,Delaware,1.203000
8,Florida,1.170893
9,Georgia,1.190857


Code for choropleth. ChatGPT assisted in finding a working JSON again.

In [14]:
# Creating choropleth for the State

state_url = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
state_response = requests.get(state_url)
state_shapes = state_response.json()

From the choropleth, it seems that Massachusetts got some really smart people :^D (Go BC!)

In [30]:
state_fig = px.choropleth_map(states_df,
                           geojson = state_shapes,
                           locations = "State",
                           featureidkey = "properties.name",
                           color = "Weighted Average",
                           center = {"lat": 42.9, "lon": -75.5},
                           zoom = 6,
                           height = 600,
                           title = "Weighted Average across the United States"
                           )
# fig.show()
HTML(state_fig.to_html(include_plotlyjs="cdn", full_html=False))

## Cleaning Dataset 2: Broadband

Coming into this dataset, the information was pretty messy. The creator of the excel decided to plug himself, and put the explanations inside the rows. This meant that I had to get rid of the jargon, and reclassify the columns

In [16]:
df_broadband

,index,Data is to be used only for analysis purposes related to broadband mapping,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,0,For questions contact bbdata@microsoft.com,NaN,NaN,NaN,NaN
1,1,NaN,NaN,NaN,NaN,NaN
2,2,Broadband availability in the United States ba...,NaN,NaN,NaN,NaN
3,3,Fourteenth Broadband Deployment Report: https:...,NaN,NaN,NaN,NaN
4,4,Form 477 data as of December 2019,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
3155,3155,WY,56037,Sweetwater County,0.9422,0.4
3156,3156,WY,56039,Teton County,0.9508,0.623
3157,3157,WY,56041,Uinta County,0.9963,0.431
3158,3158,WY,56043,Washakie County,0.8903,0.571


A lot of the clean up happens in the cell below. I dropped the extra index, removed his "Help" information, and made row 18 at the time into the column heading.

In [17]:
# Cleaning up Broadband Dataframe
df_broadband = df_broadband.drop(columns = ["index"])
df_broadband = df_broadband.iloc[17:].reset_index(drop=True)
column_header = df_broadband.iloc[0]
df_broadband.columns = column_header
df_broadband = df_broadband.drop(index = 0).reset_index(drop = True)

And below is a mini sanity check :D

In [18]:
df_broadband.head(20)

,ST,COUNTY ID,COUNTY NAME,BROADBAND AVAILABILITY PER FCC,BROADBAND USAGE
0,AL,1001,Autauga County,0.8057,0.391
1,AL,1003,Baldwin County,0.8362,0.452
2,AL,1005,Barbour County,0.6891,0.324
3,AL,1007,Bibb County,0.3368,0.136
4,AL,1009,Blount County,0.758,0.199
5,AL,1011,Bullock County,0.9363,0.157
6,AL,1013,Butler County,0.6814,0.183
7,AL,1015,Calhoun County,0.9298,0.419
8,AL,1017,Chambers County,0.8453,0.501
9,AL,1019,Cherokee County,0.9892,0.125


After looking at the cleaner dataset, I realized that I was not going to be analyzing counties this time so I decided to drop the county ID column and the county name column. I also renamed the remaining columns to make my life easier. less words typed means less mistakes for me. :DDD

In [19]:
# Only looking at Broadband and Broadband Usage
df_broadband = df_broadband.drop(columns = ["COUNTY ID", "COUNTY NAME"])
df_broadband = df_broadband.rename(columns = {"ST":"State",
                                              " BROADBAND AVAILABILITY PER FCC ":"Broadband Availability",
                                              ' BROADBAND USAGE ':'Broadband Usage'}) # renaming now to make merging easier later


Float conversion was added after I tried to combine the counties and get the means of the states. After it failed, I realized that they weren't numbers but strings. After attempting to convert it to floats the first time I realized that the creator of the excel decided to use "-" as a means to signify NA. While that's great in excel it made my life more interesting as I needed to create a filter that got rid of those pesky dashes.

In [20]:
def clean_missing(x):
    if "-" in x:
        return pd.NA
    return x

df_broadband["Broadband Availability"] = df_broadband["Broadband Availability"].apply(clean_missing)
df_broadband["Broadband Usage"] = df_broadband["Broadband Usage"].apply(clean_missing)

df_broadband = df_broadband.dropna()

In [21]:
df_broadband["Broadband Availability"]= df_broadband["Broadband Availability"].astype(float)
df_broadband["Broadband Usage"] = df_broadband["Broadband Usage"].astype(float)

From there, I converted state abbreviations to their full names, and successfully pulled means of both broadband availability and broadband usage.

In [22]:
# Changing the State Abbreviation to full state name with dictionary prior
df_broadband["State"] = df_broadband["State"].map(abbr_to_name)
broadband_avail = pd.DataFrame(df_broadband.groupby("State")["Broadband Availability"].mean()).reset_index()
broadband_usage = pd.DataFrame(df_broadband.groupby("State")["Broadband Usage"].mean()).reset_index()

Because those two means were stored in two new dataframes, I merged them together. I then dropped District of Columbia because it is not a state.

In [23]:
broadband_merge = pd.merge(broadband_avail,broadband_usage, how = "inner")
broadband_merge = broadband_merge.drop(index = 8).reset_index(drop = True)

annd the result is:

In [24]:
broadband_merge

,State,Broadband Availability,Broadband Usage
0,Alabama,0.703642,0.289597
1,Alaska,0.656067,0.351190
2,Arizona,0.749253,0.458333
3,Arkansas,0.616909,0.226627
4,California,0.922295,0.529052
5,Colorado,0.851258,0.449375
6,Connecticut,0.990563,0.668375
7,Delaware,0.976100,0.726333
8,Florida,0.839722,0.507284
9,Georgia,0.787216,0.376956


When it came to displaying the information, I thought it would be neat to compare broadband Availibilty and Broadband Usage. To do this, I needed to move the columns into rows so that Plotly could analyze it.

In [25]:
df_melt = broadband_merge.melt(id_vars = "State", 
                                value_vars = ["Broadband Availability","Broadband Usage"],
                                var_name = "Category",
                                value_name = "Value" 
                                )

In [31]:
broadband_fig = px.bar(df_melt, x = "State", y = "Value", color = "Category", barmode = "group", title = "Broadband Availability vs. Broadband Usage in the United States")
# broadband_fig.show()

HTML(broadband_fig.to_html(include_plotlyjs="cdn", full_html=False))

## Combining the Two Datasets

Now we reach the final part. Because I had made sure the formatting was good before, combining the two datasets went very smoothly. I only renamed the Weighted Average column as I thought only having "Weighted Average" on a scatter plot would be confusing.

In [27]:
# Working with the two datasets together

merged_data = pd.merge(states_df,broadband_merge,how = "outer")
merged_data = merged_data.rename(columns = {"Weighted Average":"Weighted Average of Education"})
merged_data

,State,Weighted Average of Education,Broadband Availability,Broadband Usage
0,Alabama,1.117571,0.703642,0.289597
1,Alaska,1.188000,0.656067,0.351190
2,Arizona,1.110889,0.749253,0.458333
3,Arkansas,1.124000,0.616909,0.226627
4,California,1.176038,0.922295,0.529052
5,Colorado,1.292250,0.851258,0.449375
6,Connecticut,1.298600,0.990563,0.668375
7,Delaware,1.203000,0.976100,0.726333
8,Florida,1.170893,0.839722,0.507284
9,Georgia,1.190857,0.787216,0.376956


Just a quick merge, and I could analyze the relationship between weighted average and broadband usage.

In [33]:
combined_fig = px.scatter(merged_data,
                           x = "Weighted Average of Education", 
                           y = "Broadband Usage", 
                           hover_name = "State", 
                           trendline = "ols", 
                           title = "Broadband Usage and Education (Weighted Average) Relationship by State"
                           )

# combined_fig.show()
HTML(combined_fig.to_html(include_plotlyjs="cdn", full_html=False))

## Conclusion

Based on the scatter plot and the upward-sloping regression line, the data supports my original hypothesis. States with higher broadband usage generally have a higher weighted education average. The relationship seems to be moderate-strong. It seems that having access to the internet is important when it comes to education.